### Paedawen: Training data generation

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import BedrockBatchGenerator, TrainingDataUploader
from paedacuteschema.prompt_builder import PromptBuilder
from paedacuteschema import Schema

In [ ]:
pb = PromptBuilder()

#### Download documents from batch

In [ ]:
gen = BedrockBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="paedacuteschema",
    model_name="sonnet4",
    document_batches=["paedacute-batch-2026-07-20-001.tar.gz", "normal-batch-2025-12-31-001.tar.gz", "noncancer-batch-2025-12-31-001.tar.gz"],
)

In [ ]:
gen.get_document_files_count()

#### Sanity check

##### Start batch generation

In [ ]:
gen.generate_via_batch(100, os.environ["BUCKET"],os.environ["BEDROCK_EXECUTION_ROLE"])

##### Download and parse batch outputs

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Start datagen

In [ ]:
gen.generate_via_batch(3000, os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="paedacuteschema",
    system_prompt=pb.build_main_prompt(),
    short_description="paedawen-batch",
    long_description="Training data for Paedawen",
    input_folder=Path("./data/trainingdata"),
)